<figure>
<center>
<img src="https://www.economicas.uba.ar/wp-content/uploads/2020/08/cropped-logo_FCE.png"/>
</center></figure>

# **Universidad de Buenos Aires**
## **Facultad de Ciencias Económicas**
### **Métodos Predictivos**
### Cátedra: Bianco
#### **Regresión Logística**

## Regresión Logística

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import statsmodels.api as sm
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

In [2]:
# Cargamos el dataset TITANIC directamente desde la librería seaborn
df = sns.load_dataset('titanic')

df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


**Diccionario de Variables: Dataset Titanic**

- survived: Variable Objetivo ($Y$). Indica si el pasajero sobrevivió ($1$) o no ($0$).

- pclass: Clase del pasajero. Representa el estatus socioeconómico ($1^{ra}, 2^{da}, 3^{ra}$).

- sex: Género. Variable categórica binaria (male, female).

- age: Edad. Variable numérica continua ($Age \in \mathbb{R}^{+}$).

- sibsp: Hermanos/Cónyuges. Cantidad de familiares de la misma generación a bordo ($n \in \mathbb{Z}_{\geq 0}$).

- parch: Padres/Hijos. Cantidad de familiares de distinta generación a bordo ($n \in \mathbb{Z}_{\geq 0}$).

- fare: Tarifa. Precio pagado por el ticket ($Fare > 0$).

- embarked: Puerto de embarque. Ciudad donde subió el pasajero ($C = \text{Cherbourg}$, $Q = \text{Queenstown}$, $S = \text{Southampton}$).

- class: Clase (Texto). Equivalente nominal a la clase del pasajero (First, Second, Third).

- who: Categoría biológica. Clasificación en man, woman o child.

- adult_male: Adulto Hombre. Indica si el pasajero es un varón adulto ($\text{True}/\text{False}$).

- deck: Cubierta. Letra que identifica la sección del barco ($A, B, C, D, E, F, G$).

- embark_town: Ciudad de embarque. Nombre completo del puerto de origen.

- alive: Estado de vida. Versión categórica de la supervivencia (yes, no).

- alone: Viaja solo. Indica si el pasajero no tiene familiares registrados a bordo ($\text{True}/\text{False}$).

Para nuestro modelo de Regresión Logística, buscaremos predecir la probabilidad $P(Y=1 | X)$, donde $Y$ es la columna survived y $X$ es el conjunto de variables predictoras que seleccionemos (ej: pclass, sex, age).

Se sobre entiende que las variables survived y alive es la misma variable codificada con 1 y 0 (en el primer caso) y si o no (para el segundo).
Tambien se ven otras como clase, puerto, si es hombre adulto, etc.

### Pretratamiento

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    object  
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    object  
 8   class        891 non-null    category
 9   who          891 non-null    object  
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    object  
 13  alive        891 non-null    object  
 14  alone        891 non-null    bool    
dtypes: bool(2), category(2), float64(2), int64(4), object(5)
memory usage: 80.7+ KB


In [4]:
# Verificamos cuántos nulos hay por columna
print(df.isnull().sum())

survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64


In [5]:
# Imputamos la mediana en la edad (podríamos tomar el promedio, pero la mediana suele ser más robusta y menos sensible a outliers)
df['age'] = df['age'].fillna(df['age'].median())

# Eliminamos la columna 'deck' porque tiene demasiados nulos para ser útil
df.drop(columns=['deck'], inplace=True)

# Eliminamos las filas con nulos en 'embarked' (son solo 2)
df.dropna(subset=['embarked'], inplace=True)

# Verificamos nuevamente
print(df.isnull().sum())

survived       0
pclass         0
sex            0
age            0
sibsp          0
parch          0
fare           0
embarked       0
class          0
who            0
adult_male     0
embark_town    0
alive          0
alone          0
dtype: int64


Variables que podrían traer multicolinealidad:

- alive, se duplica con survived.
- pclass con class.
- who y adult_male se duplica con sex y age.
- embark_town versión de texto de embarked
- alone, podría repetirse con sibsp

In [6]:
# Eliminamos columnas que repiten información y podrían traer multicolinealidad.
cols_to_drop = ['alive', 'class', 'who', 'adult_male', 'embark_town', 'alone']
df.drop(columns=cols_to_drop, inplace=True)
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


In [7]:
# Convertimos 'sex' y 'embarked' en variables numéricas
# drop_first=True evita la "trampa de la variable ficticia" (correlación perfecta)
df = pd.get_dummies(df, columns=['sex', 'embarked'], drop_first=True)
df.head()

,survived,pclass,age,sibsp,parch,fare,sex_male,embarked_Q,embarked_S
0,0,3,22.0,1,0,7.2500,True,False,True
1,1,1,38.0,1,0,71.2833,False,False,False
2,1,3,26.0,0,0,7.9250,False,False,True
3,1,1,35.0,1,0,53.1000,False,False,True
4,0,3,35.0,0,0,8.0500,True,False,True


**Nota:**

**Si el objetivo es el Análisis de Inferencia (Explicar el fenómeno):** No es estrictamente necesario separar el modelo en train y test. Cuando queremos entender cómo afectan las variables (edad, clase, género) a la probabilidad de sobrevivir de esos pasajeros en particular, usamos el 100% de los datos para que las estimaciones de los coeficientes ($\beta$) y los p-values sean lo más precisas posibles.

**Si el objetivo es la Predicción (Validar el modelo):** Es obligatorio separar. Necesitamos saber si el modelo que aprendió con los datos del Titanic funcionaría igual de bien con pasajeros "nuevos" que el modelo nunca vio.

Definición de Matriz de Características ($X$) y Vector Objetivo ($y$)

Ahora separamos los datos para que el modelo sepa qué tiene que predecir.

In [8]:
# y es lo que queremos predecir (Sobrevivió o no)
y = df['survived']

# X son nuestros predictores (todas las demás columnas)
X = df.drop(columns=['survived'])

print("Variables predictoras finales:", X.columns.tolist())

Variables predictoras finales: ['pclass', 'age', 'sibsp', 'parch', 'fare', 'sex_male', 'embarked_Q', 'embarked_S']


In [9]:
# 1. Agregamos la columna de constante (el intercepto del modelo), este es un paso necesario en statsmodels para que el informe sea completo
X_with_const = sm.add_constant(X)

# 2. Definimos y entrenamos el modelo Logit
modelo_stats = sm.Logit(y, X_with_const.astype(float)) # Aseguramos que sea float
resultado = modelo_stats.fit()

Optimization terminated successfully.
         Current function value: 0.441471
         Iterations 6


In [10]:
# 3. Mostramos el informe
print(resultado.summary())

                           Logit Regression Results                           
Dep. Variable:               survived   No. Observations:                  889
Model:                          Logit   Df Residuals:                      880
Method:                           MLE   Df Model:                            8
Date:                Sat, 30 May 2026   Pseudo R-squ.:                  0.3364
Time:                        22:28:28   Log-Likelihood:                -392.47
converged:                       True   LL-Null:                       -591.41
Covariance Type:            nonrobust   LLR p-value:                 5.314e-81
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          5.2469      0.562      9.336      0.000       4.145       6.348
pclass        -1.0987      0.144     -7.654      0.000      -1.380      -0.817
age           -0.0393      0.008     -5.013      0.0

**Diagnóstico Global del Modelo**

Antes de mirar las variables, miramos si el modelo como un todo tiene sentido:

- Pseudo R-squ. ($0.3364$): Como vimos antes, en logística un valor entre $0.2$ y $0.4$ indica un ajuste muy robusto. El modelo explica bien la supervivencia.
- LLR p-value ($5.314e-81$): Es un número virtualmente cero. El modelo es globalmente significativo. No es fruto del azar.

**Análisis de variables**

*Las que SÍ aportan (Significativas $p < 0.05$):*

- const ($0.000$): El punto de partida matemático.
- pclass ($0.000$): Fundamental. A medida que aumenta el número de clase (ej. de 1ra a 3ra), la probabilidad de sobrevivir baja (coeficiente negativo $-1.09$).
- age ($0.000$): Significativa. A mayor edad, menor probabilidad de sobrevivir (coeficiente $-0.039$).
- sex_male ($0.000$): La más pesada. Ser hombre disminuye drásticamente la probabilidad de sobrevivir (coeficiente $-2.71$).
- sibsp ($0.003$): Significativa. Tener muchos hermanos/cónyuges a bordo redujo las chances.

*Las que NO aportan (No significativas $p > 0.05$):*

- parch ($0.454$): No influye estadísticamente en este modelo.
- fare ($0.414$): una vez que ya sabemos la pclass, el precio exacto del ticket no agrega información nueva. Probar que sucedería si no tenemos la clase y sí el precio de la tarifa.
- embarked_Q y embarked_S: El puerto de salida no parece ser un factor determinante para sobrevivir (p-values de $0.86$ y $0.08$).

Un nuevo modelo con las variables significativas.

In [11]:
# 1. Seleccionamos solo las variables significativas
variables_limpias = ['pclass', 'age', 'sibsp', 'sex_male']
X_clean = X[variables_limpias]

# 2. Agregamos la constante
X_clean_const = sm.add_constant(X_clean)

# 3. Re-entrenamos el modelo
modelo_final = sm.Logit(y, X_clean_const.astype(float)).fit()

# 4. Mostramos los resultados finales
print(modelo_final.summary())

Optimization terminated successfully.
         Current function value: 0.444704
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:               survived   No. Observations:                  889
Model:                          Logit   Df Residuals:                      884
Method:                           MLE   Df Model:                            4
Date:                Sat, 30 May 2026   Pseudo R-squ.:                  0.3315
Time:                        22:28:28   Log-Likelihood:                -395.34
converged:                       True   LL-Null:                       -591.41
Covariance Type:            nonrobust   LLR p-value:                 1.392e-83
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          5.1702      0.477     10.831      0.000       4.235       6.106
pclass        -1.1727      0.

- La Barrera Social: Cada salto en pclass (de 1ra a 2da, o de 2da a 3ra) redujo significativamente las chances de vivir. No era solo un ticket, era una posición en el barco.
- La Vulnerabilidad de la Edad: Aunque significativa, su impacto por cada año es pequeño ($-0.039$). Sin embargo, en la suma total (un niño vs. un anciano), la diferencia se vuelve crítica.
- Red de Contención: sibsp nos dice que viajar con demasiada familia directa pudo haber dificultado la movilidad o la toma de decisiones rápidas durante la evacuación.
- El Factor Dominante: El género (sex_male). "Mujeres y niños primero" tiene una base estadística real aquí.

Supongamos un caso muy común en el Titanic:

Nombre: "Juan Pérez"

Clase: 3ra clase (pclass = 3)

Edad: 30 años (age = 30)

Compañía: Viaja solo (sibsp = 0)

Sexo: Hombre (sex_male = 1)

In [12]:
# 1. Definimos los datos del nuevo pasajero (incluyendo el 1 para la constante)
# [const, pclass, age, sibsp, sex_male]
pasajero_juan = [1, 3, 30, 0, 1]

# 2. Realizamos la predicción usando el modelo entrenado
probabilidad = modelo_final.predict(pasajero_juan)[0]

print(f"La probabilidad de supervivencia de Juan es: {probabilidad:.2%}")

La probabilidad de supervivencia de Juan es: 9.33%


El Cálculo Manual

La fórmula de la combinación lineal es:

$$z = \text{const} + (\beta_{1} \cdot \text{pclass}) + (\beta_{2} \cdot \text{age}) + (\beta_{3} \cdot \text{sibsp}) + (\beta_{4} \cdot \text{sex\_male})$$

Sustituimos con los valores del reporte:

$z = 5.1702 + (-1.1727 \cdot 3) + (-0.0398 \cdot 30) + (-0.3534 \cdot 0) + (-2.7322 \cdot 1)$

$z = 5.1702 - 3.5181 - 1.194 - 0 - 2.7322$

$z = -2.2741$

Ahora aplicamos la función logística para obtener la probabilidad ($P$):

$$P = \frac{1}{1 + e^{-z}} = \frac{1}{1 + e^{2.2741}} \approx \mathbf{0.093}$$

Resultado: Juan tiene solo un 9.3% de probabilidad de sobrevivir.

In [13]:
pasajero_jack = [1, 3, 20, 0, 1]

# 2. Realizamos la predicción usando el modelo entrenado
probabilidad = modelo_final.predict(pasajero_jack)[0]

print(f"La probabilidad de supervivencia de Jack es: {probabilidad:.2%}")

La probabilidad de supervivencia de Jack es: 13.29%


In [14]:
pasajero_rose = [1, 1, 17, 1, 0]

# 2. Realizamos la predicción usando el modelo entrenado
probabilidad = modelo_final.predict(pasajero_rose)[0]

print(f"La probabilidad de supervivencia de Rose es: {probabilidad:.2%}")

La probabilidad de supervivencia de Rose es: 95.11%


Veamos la matriz de confusión:

In [15]:
# Obtenemos las probabilidades del modelo limpio. Como si no tuvieramos el dato real.
predicciones_prob = modelo_final.predict(X_clean_const.astype(float))
predicciones_clase = (predicciones_prob > 0.5).astype(int) # se redondean las probabilidades.

# Creamos la matriz de confusión
matriz = confusion_matrix(y, predicciones_clase)

# La convertimos a DataFrame
matriz_df = pd.DataFrame(matriz,
                         index=['Real: No Sobrevive (0)', 'Real: Sobrevive (1)'],
                         columns=['Predicho: No Sobrevive (0)', 'Predicho: Sobrevive (1)'])

print(matriz_df)

                        Predicho: No Sobrevive (0)  Predicho: Sobrevive (1)
Real: No Sobrevive (0)                         459                       90
Real: Sobrevive (1)                             99                      241


Los Cálculos Manuales

Accuracy (Exactitud): Es el porcentaje de predicciones correctas sobre el total.

$$\text{Accuracy} = \frac{TP + TN}{Total} = \frac{238 + 475}{889} = \frac{713}{889} \approx \mathbf{0.802} \text{ (80.2\%)}$$

Precision (Precisión): De todos los que el modelo marcó como "Sobrevive", ¿cuántos realmente lo hicieron? (Mide la calidad de la alarma).

$$\text{Precision} = \frac{TP}{TP + FP} = \frac{238}{238 + 74} = \frac{238}{312} \approx \mathbf{0.762} \text{ (76.2\%)}$$

Recall (Sensibilidad/Exhaustividad): De todas las personas que realmente sobrevivieron, ¿a cuántas logramos detectar? (Mide la cobertura).

$$\text{Recall} = \frac{TP}{TP + FN} = \frac{238}{238 + 102} = \frac{238}{340} \approx \mathbf{0.700} \text{ (70.0\%)}$$

In [16]:
# Calculamos las métricas
acc = accuracy_score(y, predicciones_clase)
prec = precision_score(y, predicciones_clase)
rec = recall_score(y, predicciones_clase)

# Mostramos los resultados
print(f"Exactitud (Accuracy): {acc:.2%}")
print(f"Precisión (Precision): {prec:.2%}")
print(f"Sensibilidad (Recall): {rec:.2%}")

Exactitud (Accuracy): 78.74%
Precisión (Precision): 72.81%
Sensibilidad (Recall): 70.88%


Repetir el ejercicio con el dataset de diabetes.